In [57]:
import torch
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
import random

In [87]:
words = open("names.txt" ,'r').read().splitlines()
len(words)
#vocaublary from words
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] =0
itos = {i:s for s,i in stoi.items()}
itos
voab_size = len(stoi)
print(voab_size)

27


In [126]:
block_size =8
def build_dataset(words):
    X,Y=[],[]

    for w in words:
        context = [0]*block_size       
        for i in w+'.':
            X.append([i for i in context])
            Y.append(stoi[i])
            context = context[1:]+[stoi[i]]
    
    x= torch.tensor(X)
    y=torch.tensor(Y)
    return x,y


# x,y =build_dataset(words[:3])
# print(x)
# print(len(y))   
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

xtr , ytr = build_dataset(words[:n1]) 
xval , yval = build_dataset(words[n1:n2]) 
xtest , ytest = build_dataset(words[n2:])     
        
    

In [140]:
# Near copy paste of the layers we have developed in Part 3

# -----------------------------------------------------------------------------------------------
class Linear:
  
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out)) / fan_in**0.5 # note: kaiming init
    self.bias = torch.zeros(fan_out) if bias else None
  
  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out
  
  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])

# -----------------------------------------------------------------------------------------------
class BatchNorm1d:
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)
  
  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      if x.ndim == 2:
        dim = 0
      elif x.ndim == 3:
        dim = (0,1)
      xmean = x.mean(dim, keepdim=True) # batch mean
      xvar = x.var(dim, keepdim=True) # batch variance
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    # update the buffers
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

# -----------------------------------------------------------------------------------------------
class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []
  
  
class Embedding:
  def __init__(self , num_embedding , embedding_dim):
    self.weights = torch.randn((num_embedding,embedding_dim))
  
  def __call__(self, IX):
    self.out = self.weights[IX]
    return self.out
  
  def parameters(self) :
    return [self.weights]
  

class FlattenConsecutive:
  def __init__(self,n):
    self.n =n
  
  def __call__(self, x):
    B,T,C = x.shape
    x = x.view(B,T//self.n,C*self.n)
    if x.shape[1] ==1:
      x =x.squeeze(1)
    self.out = x
    return self.out
  
  def parameters(self):
    return []
    
class Sequintial:
  def __init__(self ,layers):
    self.layers =layers
  
  def __call__(self, x):
    for layer in self.layers:
        x =layer(x)
    self.out = x
    return self.out
  
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]
  
    


In [128]:
torch.manual_seed(42)

In [ ]:
n_embd =10
n_hidden=200

model =Sequintial([
    Embedding(voab_size,n_embd),
    FlattenConsecutive(2),Linear(n_embd*2,n_hidden ,bias=True) , BatchNorm1d(n_hidden) , Tanh(),
    FlattenConsecutive(2),Linear(n_hidden*2,n_hidden ,bias=True) , BatchNorm1d(n_hidden) , Tanh(),
        Linear(n_hidden,voab_size)
        ])



In [154]:

with torch.no_grad():
    model.layers[-1].weight *=0.1
parameters = model.parameters()


for p in parameters:
    p.requires_grad =True
    

In [155]:
max_steps = 200000
batch_size =32
lossi =[]

for i in range(max_steps):
    
    ix = torch.randint(0, xtr.shape[0],(batch_size,))
    xb,yb = xtr[ix],ytr[ix]
    

    x= xb
    
    logits = model(x)
    
    loss = F.cross_entropy(logits,yb)
    
    for p in parameters:
        p.grad = None
    
    loss.backward()
    
    lr = 0.1 if i <150000 else 0.01
    for p in parameters :
        p.data += -lr * p.grad
    
    
    if i% 10000 ==0:
        print(f'{i} --> loss {loss.item()}')
    lossi.append(loss.log10().item())
    break
    
        

RuntimeError: Expected target size [32, 27], got [32]

In [149]:
for layer in model.layers:
    print(layer.__class__.__name__ ,'--->', layer.out.shape)

Embedding ---> torch.Size([32, 8, 10])
FlattenConsecutive ---> torch.Size([32, 80])
Linear ---> torch.Size([32, 200])
BatchNorm1d ---> torch.Size([32, 200])
Tanh ---> torch.Size([32, 200])
Linear ---> torch.Size([32, 27])


In [132]:
for layer in model.layers:
    layer.training = False

In [133]:
@torch.no_grad()
def split_loss(split):
    x,y ={
        'train':(xtr,ytr)
        ,'val': (xval,yval)
        ,'test': (xtest,ytest)
    }[split]
    logits =model(x)
    loss  = F.cross_entropy(logits,y)
    print(split ,loss.item())
    
split_loss("train") 
split_loss("val") 

train 1.920793056488037
val 2.0282320976257324


In [139]:

for _ in range(20):
    out  =[]
    context =[0]*block_size
    # torch.tensor(context).shape ,torch.tensor([context]).shape
    while True:
    
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits,dim=1)
        ix = torch.multinomial(probs,num_samples=1).item()
        out.append(itos[ix])
        context=context[1:]+[ix]
        if ix ==0: 
            break
    print(''.join([i for i in out]))

        
     
            



caydence.
elas.
lyrah.
kenny.
mordofette.
jayonne.
kamella.
khayson.
adelis.
lovalee.
ellisa.
maybon.
alibeth.
joleen.
dilyn.
rhipo.
blatis.
cinsla.
ved.
oliva.
